In [1]:
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

# Target
target = "satisfaction"

# Features
drop_cols = [target, "id"]

X = train.drop(columns=drop_cols).copy()
y = train[target].astype(int)

X_test = test.drop(columns=["id"]).copy()

# Categorical columns
cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

# Same validation split for every model
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)
print("Positive rate:", y.mean())

Train: (559708, 21)
Valid: (139927, 21)
Test : (299844, 21)
Positive rate: 0.44357272006117476


In [4]:
# ============================================================
# PHASE 3A — LOAD ALL SAVED PREDICTIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# Load validation predictions
# ------------------------------------------------------------

cat_valid = np.load("artifacts/oof/baseline_cat.npy")
lgb_valid = np.load("artifacts/oof/baseline_lgb.npy")
xgb_valid = np.load("artifacts/oof/baseline_xgb.npy")
lgb_opt_valid = np.load("artifacts/oof/lgb_optimized.npy")

# ------------------------------------------------------------
# Load test predictions
# ------------------------------------------------------------

cat_test = np.load("artifacts/test/baseline_cat.npy")
lgb_test = np.load("artifacts/test/baseline_lgb.npy")
xgb_test = np.load("artifacts/test/baseline_xgb.npy")
lgb_opt_test = np.load("artifacts/test/lgb_optimized.npy")

# ------------------------------------------------------------
# Verify shapes
# ------------------------------------------------------------

print("VALIDATION SHAPES")
print("CatBoost :", cat_valid.shape)
print("LightGBM :", lgb_valid.shape)
print("XGBoost  :", xgb_valid.shape)
print("LGB Opt  :", lgb_opt_valid.shape)

print("\nTEST SHAPES")
print("CatBoost :", cat_test.shape)
print("LightGBM :", lgb_test.shape)
print("XGBoost  :", xgb_test.shape)
print("LGB Opt  :", lgb_opt_test.shape)

assert len(cat_valid) == len(y_valid)
assert len(lgb_valid) == len(y_valid)
assert len(xgb_valid) == len(y_valid)
assert len(lgb_opt_valid) == len(y_valid)

assert len(cat_test) == len(X_test)
assert len(lgb_test) == len(X_test)
assert len(xgb_test) == len(X_test)
assert len(lgb_opt_test) == len(X_test)

print("\n✓ All prediction arrays align correctly.")

VALIDATION SHAPES
CatBoost : (139927,)
LightGBM : (139927,)
XGBoost  : (139927,)
LGB Opt  : (139927,)

TEST SHAPES
CatBoost : (299844,)
LightGBM : (299844,)
XGBoost  : (299844,)
LGB Opt  : (299844,)

✓ All prediction arrays align correctly.


In [5]:
# ============================================================
# INDIVIDUAL MODEL VERIFICATION
# ============================================================

individual_scores = {
    "CatBoost baseline": roc_auc_score(y_valid, cat_valid),
    "LightGBM baseline": roc_auc_score(y_valid, lgb_valid),
    "XGBoost baseline": roc_auc_score(y_valid, xgb_valid),
    "LightGBM optimized": roc_auc_score(y_valid, lgb_opt_valid),
}

scores_df = (
    pd.DataFrame(
        individual_scores.items(),
        columns=["Model", "AUC"]
    )
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

print("=" * 70)
print("SAVED PREDICTION VERIFICATION")
print("=" * 70)
print(scores_df.to_string(index=False))

SAVED PREDICTION VERIFICATION
             Model      AUC
LightGBM optimized 0.958135
 LightGBM baseline 0.957875
  XGBoost baseline 0.957796
 CatBoost baseline 0.957698


In [6]:
# ============================================================
# PREDICTION CORRELATION
# ============================================================

pred_df = pd.DataFrame({
    "CatBoost": cat_valid,
    "LightGBM": lgb_valid,
    "XGBoost": xgb_valid,
    "LightGBM_Optimized": lgb_opt_valid
})

corr = pred_df.corr()

print("=" * 70)
print("PREDICTION CORRELATION")
print("=" * 70)
print(corr.round(6))

PREDICTION CORRELATION
                    CatBoost  LightGBM   XGBoost  LightGBM_Optimized
CatBoost            1.000000  0.997972  0.997875            0.997777
LightGBM            0.997972  1.000000  0.997846            0.999379
XGBoost             0.997875  0.997846  1.000000            0.997929
LightGBM_Optimized  0.997777  0.999379  0.997929            1.000000


In [7]:
train["Arrival_Delay_Missing"] = (
    train["Arrival Delay in Minutes"].isna().astype(int)
)

test["Arrival_Delay_Missing"] = (
    test["Arrival Delay in Minutes"].isna().astype(int)
)

In [8]:
# ============================================================
# MISSINGNESS ANALYSIS
# ============================================================

missing_mask = train["Arrival Delay in Minutes"].isna()

print("Missing rows:", missing_mask.sum())
print("Missing %:", missing_mask.mean() * 100)

print("\nTarget distribution:")
print(
    train.groupby(missing_mask)["satisfaction"]
    .agg(["count", "mean"])
)

Missing rows: 292
Missing %: 0.041736048082214296

Target distribution:
                           count      mean
Arrival Delay in Minutes                  
False                     699343  0.443606
True                         292  0.363014


In [9]:
Arrival_Delay_Missing = train["Arrival Delay in Minutes"].isna().astype(int)
missing_valid = X_valid["Arrival Delay in Minutes"].isna()

print("Validation missing rows:", missing_valid.sum())

if missing_valid.sum() > 0:
    print(
        y_valid[missing_valid].mean()
    )

Validation missing rows: 65
0.38461538461538464


In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ============================================================
# LOAD VALIDATION PREDICTIONS
# ============================================================

cat_valid = np.load("artifacts/oof/baseline_cat.npy")
lgb_valid = np.load("artifacts/oof/lgb_optimized.npy")
xgb_valid = np.load("artifacts/oof/baseline_xgb.npy")

# ============================================================
# VERIFY INDIVIDUAL AUCs
# ============================================================

print("=" * 70)
print("INDIVIDUAL MODEL AUCs")
print("=" * 70)

auc_cat = roc_auc_score(y_valid, cat_valid)
auc_lgb = roc_auc_score(y_valid, lgb_valid)
auc_xgb = roc_auc_score(y_valid, xgb_valid)

print(f"CatBoost       : {auc_cat:.9f}")
print(f"LightGBM Opt   : {auc_lgb:.9f}")
print(f"XGBoost        : {auc_xgb:.9f}")

# ============================================================
# PREDICTION CORRELATION
# ============================================================

pred_df = pd.DataFrame({
    "LightGBM": lgb_valid,
    "XGBoost": xgb_valid,
    "CatBoost": cat_valid
})

print("\n" + "=" * 70)
print("PREDICTION CORRELATION")
print("=" * 70)
print(pred_df.corr())

# ============================================================
# COARSE 5% WEIGHT SEARCH
# ============================================================

print("\n" + "=" * 70)
print("COARSE BLEND SEARCH — 5% GRID")
print("=" * 70)

results = []

for w_lgb in np.arange(0.0, 1.001, 0.05):
    for w_xgb in np.arange(0.0, 1.001 - w_lgb, 0.05):

        w_cat = 1.0 - w_lgb - w_xgb

        blend = (
            w_lgb * lgb_valid
            + w_xgb * xgb_valid
            + w_cat * cat_valid
        )

        auc = roc_auc_score(y_valid, blend)

        results.append({
            "LGB": w_lgb,
            "XGB": w_xgb,
            "CAT": w_cat,
            "AUC": auc
        })

results_df = (
    pd.DataFrame(results)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nTOP 15 BLENDS:")
print(results_df.head(15).to_string(index=False))

# ============================================================
# COMPARE AGAINST BEST SINGLE MODEL
# ============================================================

best = results_df.iloc[0]

print("\n" + "=" * 70)
print("BEST COARSE BLEND")
print("=" * 70)

print(f"LGB weight : {best['LGB']:.2f}")
print(f"XGB weight : {best['XGB']:.2f}")
print(f"CAT weight : {best['CAT']:.2f}")
print(f"Blend AUC  : {best['AUC']:.9f}")

print("\nImprovement vs optimized LGB:")
print(f"{best['AUC'] - auc_lgb:+.9f}")

INDIVIDUAL MODEL AUCs
CatBoost       : 0.957697739
LightGBM Opt   : 0.958134780
XGBoost        : 0.957796183

PREDICTION CORRELATION
          LightGBM   XGBoost  CatBoost
LightGBM  1.000000  0.997929  0.997777
XGBoost   0.997929  1.000000  0.997875
CatBoost  0.997777  0.997875  1.000000

COARSE BLEND SEARCH — 5% GRID

TOP 15 BLENDS:
 LGB  XGB  CAT      AUC
0.55 0.25 0.20 0.958338
0.60 0.25 0.15 0.958337
0.60 0.20 0.20 0.958336
0.55 0.30 0.15 0.958335
0.50 0.30 0.20 0.958334
0.50 0.25 0.25 0.958334
0.55 0.20 0.25 0.958334
0.65 0.20 0.15 0.958332
0.60 0.30 0.10 0.958329
0.65 0.25 0.10 0.958329
0.45 0.30 0.25 0.958329
0.50 0.35 0.15 0.958327
0.65 0.15 0.20 0.958327
0.60 0.15 0.25 0.958326
0.50 0.20 0.30 0.958326

BEST COARSE BLEND
LGB weight : 0.55
XGB weight : 0.25
CAT weight : 0.20
Blend AUC  : 0.958338254

Improvement vs optimized LGB:
+0.000203474


In [11]:
# ============================================================
# PHASE 3B — FINE 1% BLEND SEARCH
# ============================================================

fine_results = []

for w_lgb in np.arange(0.45, 0.651, 0.01):
    for w_xgb in np.arange(0.15, 0.351, 0.01):

        w_cat = 1.0 - w_lgb - w_xgb

        # Valid weights only
        if w_cat < 0.10 or w_cat > 0.40:
            continue

        blend = (
            w_lgb * lgb_valid
            + w_xgb * xgb_valid
            + w_cat * cat_valid
        )

        auc = roc_auc_score(y_valid, blend)

        fine_results.append({
            "LGB": w_lgb,
            "XGB": w_xgb,
            "CAT": w_cat,
            "AUC": auc
        })

fine_results_df = (
    pd.DataFrame(fine_results)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

print("=" * 70)
print("TOP 20 FINE BLENDS")
print("=" * 70)

print(
    fine_results_df.head(20).to_string(index=False)
)

best_fine = fine_results_df.iloc[0]

print("\n" + "=" * 70)
print("BEST FINE BLEND")
print("=" * 70)

print(f"LGB weight : {best_fine['LGB']:.2%}")
print(f"XGB weight : {best_fine['XGB']:.2%}")
print(f"CAT weight : {best_fine['CAT']:.2%}")
print(f"AUC        : {best_fine['AUC']:.9f}")

print("\nImprovement vs optimized LGB:")
print(f"{best_fine['AUC'] - auc_lgb:+.9f}")

print("\nImprovement vs coarse blend:")
print(f"{best_fine['AUC'] - 0.958338254:+.9f}")

TOP 20 FINE BLENDS
 LGB  XGB  CAT      AUC
0.57 0.24 0.19 0.958338
0.56 0.25 0.19 0.958338
0.57 0.25 0.18 0.958338
0.55 0.25 0.20 0.958338
0.57 0.23 0.20 0.958338
0.56 0.24 0.20 0.958338
0.58 0.23 0.19 0.958338
0.55 0.26 0.19 0.958338
0.58 0.24 0.18 0.958338
0.56 0.26 0.18 0.958338
0.54 0.26 0.20 0.958338
0.59 0.23 0.18 0.958338
0.56 0.23 0.21 0.958338
0.54 0.25 0.21 0.958338
0.54 0.27 0.19 0.958338
0.55 0.24 0.21 0.958338
0.58 0.22 0.20 0.958338
0.55 0.27 0.18 0.958338
0.57 0.26 0.17 0.958338
0.53 0.26 0.21 0.958338

BEST FINE BLEND
LGB weight : 57.00%
XGB weight : 24.00%
CAT weight : 19.00%
AUC        : 0.958338417

Improvement vs optimized LGB:
+0.000203637

Improvement vs coarse blend:
+0.000000163


In [12]:
# ============================================================
# FINAL LOCKED BLEND
# ============================================================

cat_test = np.load("artifacts/test/baseline_cat.npy")
lgb_test = np.load("artifacts/test/lgb_optimized.npy")
xgb_test = np.load("artifacts/test/baseline_xgb.npy")

FINAL_LGB_WEIGHT = 0.57
FINAL_XGB_WEIGHT = 0.24
FINAL_CAT_WEIGHT = 0.19

final_test_pred = (
    FINAL_LGB_WEIGHT * lgb_test
    + FINAL_XGB_WEIGHT * xgb_test
    + FINAL_CAT_WEIGHT * cat_test
)

print("Prediction count:", len(final_test_pred))
print("Min:", final_test_pred.min())
print("Max:", final_test_pred.max())
print("Mean:", final_test_pred.mean())

Prediction count: 299844
Min: 0.0034546445489461197
Max: 0.9916668134159493
Mean: 0.4434203334848861


In [14]:
os.makedirs("artifacts/submissions", exist_ok=True)

In [15]:
submission = pd.DataFrame({
    "id": test["id"],
    "satisfaction": final_test_pred
})

submission_path = "artifacts/submissions/submission_blend_lgb57_xgb24_cat19.csv"

submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Shape:", submission.shape)
print(submission.head())

Saved: artifacts/submissions/submission_blend_lgb57_xgb24_cat19.csv
Shape: (299844, 2)
       id  satisfaction
0  699635      0.238800
1  699636      0.873806
2  699637      0.919572
3  699638      0.069043
4  699639      0.047479


### score => **0.95857**  rank => **338**